In [2]:
import os
import warnings
import numpy as np
import pandas as pd

from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")


In [3]:
FEATURE_FOLDER = "feature_output"

files = sorted([
    f for f in os.listdir(FEATURE_FOLDER)
    if f.endswith("_features.csv")
])

dfs = []
for f in files:
    df = pd.read_csv(os.path.join(FEATURE_FOLDER, f), parse_dates=["hour"])
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Shape :", data.shape)
print("\nDatasets:")
print(data["source_dataset"].value_counts())

data.head()


Shape : (261090, 44)

Datasets:
source_dataset
palo_alto    82730
acn          66020
perth        34312
sap          31443
boulder      28441
dundee       16704
paris         1440
Name: count, dtype: int64


,hour,source_dataset,entity_id,site,region,kwh_allocated,hour_of_day,day_of_week,month,is_weekend,...,roll_mean_168h,roll_std_168h,trend_1h,accel_1h,trend_24h,prev_day_peak,prev_week_peak,consec_zero_hours,consec_active_hours,volatility_24h
0,2018-04-25 04:00:00+00:00,acn,caltech,caltech,California,3.118359,21,1,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
1,2018-04-25 05:00:00+00:00,acn,caltech,caltech,California,3.602725,22,1,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,NaN
2,2018-04-25 06:00:00+00:00,acn,caltech,caltech,California,1.565481,23,1,4,0,...,NaN,NaN,0.484366,NaN,NaN,NaN,NaN,0,2,NaN
3,2018-04-25 07:00:00+00:00,acn,caltech,caltech,California,3.163750,0,2,4,0,...,NaN,NaN,-2.037244,-2.521611,NaN,NaN,NaN,0,3,NaN
4,2018-04-25 08:00:00+00:00,acn,caltech,caltech,California,14.703062,1,2,4,0,...,NaN,NaN,1.598269,3.635514,NaN,NaN,NaN,0,4,NaN


In [4]:
TARGET = "kwh_allocated"

data = data.sort_values(["source_dataset", "hour"]).reset_index(drop=True)
data = data.dropna().reset_index(drop=True)

print("Final Shape:", data.shape)


Final Shape: (259578, 44)


In [5]:
hurdle_data = data.sort_values(["entity_id", "hour"]).reset_index(drop=True)

train_list, val_list, test_list = [], [], []

for _, group in hurdle_data.groupby("entity_id"):
    group = group.sort_values("hour")
    n = len(group)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    train_list.append(group.iloc[:train_end])
    val_list.append(group.iloc[train_end:val_end])
    test_list.append(group.iloc[val_end:])

train_df = pd.concat(train_list).reset_index(drop=True)
val_df = pd.concat(val_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)

print(train_df.shape, val_df.shape, test_df.shape)


(181701, 44) (38936, 44) (38941, 44)


In [6]:
entity_stats = (
    train_df.groupby("entity_id")["kwh_allocated"]
    .agg(entity_mean_kwh="mean", entity_std_kwh="std", entity_max_kwh="max")
    .reset_index()
)
entity_stats["entity_std_kwh"] = entity_stats["entity_std_kwh"].fillna(0)

train_df = train_df.merge(entity_stats, on="entity_id", how="left")
val_df = val_df.merge(entity_stats, on="entity_id", how="left")
test_df = test_df.merge(entity_stats, on="entity_id", how="left")

global_mean = train_df["kwh_allocated"].mean()
for d in [train_df, val_df, test_df]:
    d["entity_mean_kwh"] = d["entity_mean_kwh"].fillna(global_mean)
    d["entity_std_kwh"] = d["entity_std_kwh"].fillna(0)
    d["entity_max_kwh"] = d["entity_max_kwh"].fillna(global_mean)

print(train_df[["entity_mean_kwh", "entity_std_kwh", "entity_max_kwh"]].describe())


       entity_mean_kwh  entity_std_kwh  entity_max_kwh
count    181701.000000   181701.000000   181701.000000
mean         16.775782       23.638648      138.203260
std           6.417672       10.052966       64.157822
min           1.576295        3.484430       31.480456
25%          13.074826       15.865517       87.452504
50%          17.902909       25.889754      139.267655
75%          22.213566       29.892553      160.526648
max          33.551955       37.774129      270.398046


In [7]:
DROP_COLUMNS = ["hour", "kwh_allocated", "entity_id", "site"]

X_train = train_df.drop(columns=DROP_COLUMNS)
y_train = train_df[TARGET]

X_val = val_df.drop(columns=DROP_COLUMNS)
y_val = val_df[TARGET]

X_test = test_df.drop(columns=DROP_COLUMNS)
y_test = test_df[TARGET]

categorical_cols = [c for c in ["source_dataset", "region"] if c in X_train.columns]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_val = pd.get_dummies(X_val, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print(X_train.shape, X_val.shape, X_test.shape)


(181701, 51) (38936, 51) (38941, 51)


In [8]:
y_train_binary = (y_train > 0).astype(int)
y_val_binary = (y_val > 0).astype(int)
y_test_binary = (y_test > 0).astype(int)

print("Share of nonzero hours - train:", y_train_binary.mean())
print("Share of nonzero hours - val:  ", y_val_binary.mean())
print("Share of nonzero hours - test: ", y_test_binary.mean())

clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

clf.fit(
    X_train, y_train_binary,
    eval_set=[(X_train, y_train_binary), (X_val, y_val_binary)],
    verbose=50,
)


Share of nonzero hours - train: 0.7129735114281154
Share of nonzero hours - val:   0.6476268748715841
Share of nonzero hours - test:  0.651857938933258
[0]	validation_0-logloss:0.55410	validation_1-logloss:0.60707
[50]	validation_0-logloss:0.07235	validation_1-logloss:0.07694
[100]	validation_0-logloss:0.04195	validation_1-logloss:0.04633
[150]	validation_0-logloss:0.03758	validation_1-logloss:0.04299
[200]	validation_0-logloss:0.03515	validation_1-logloss:0.04212
[250]	validation_0-logloss:0.03304	validation_1-logloss:0.04201
[300]	validation_0-logloss:0.03106	validation_1-logloss:0.04185
[350]	validation_0-logloss:0.02953	validation_1-logloss:0.04190
[353]	validation_0-logloss:0.02945	validation_1-logloss:0.04189


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,30
,enable_categorical,True
,eval_metric,'logloss'


In [9]:
nonzero_train_mask = (y_train > 0).values
nonzero_val_mask = (y_val > 0).values

X_train_nz = X_train[nonzero_train_mask]
y_train_nz = y_train[nonzero_train_mask]

X_val_nz = X_val[nonzero_val_mask]
y_val_nz = y_val[nonzero_val_mask]

reg = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:tweedie",
    tweedie_variance_power=1.3,
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

reg.fit(
    X_train_nz, y_train_nz,
    eval_set=[(X_train_nz, y_train_nz), (X_val_nz, y_val_nz)],
    verbose=50,
)


[0]	validation_0-tweedie-nloglik@1.3:43.01020	validation_1-tweedie-nloglik@1.3:66.38222
[50]	validation_0-tweedie-nloglik@1.3:38.35667	validation_1-tweedie-nloglik@1.3:58.39194
[100]	validation_0-tweedie-nloglik@1.3:38.25409	validation_1-tweedie-nloglik@1.3:58.26421
[150]	validation_0-tweedie-nloglik@1.3:38.23714	validation_1-tweedie-nloglik@1.3:58.25181
[200]	validation_0-tweedie-nloglik@1.3:38.22660	validation_1-tweedie-nloglik@1.3:58.24397
[250]	validation_0-tweedie-nloglik@1.3:38.21723	validation_1-tweedie-nloglik@1.3:58.24307
[300]	validation_0-tweedie-nloglik@1.3:38.20979	validation_1-tweedie-nloglik@1.3:58.23963
[350]	validation_0-tweedie-nloglik@1.3:38.20324	validation_1-tweedie-nloglik@1.3:58.23758
[397]	validation_0-tweedie-nloglik@1.3:38.19693	validation_1-tweedie-nloglik@1.3:58.24124


,objective,'reg:tweedie'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,30
,enable_categorical,True
,eval_metric,None


In [10]:
def hurdle_predict(clf_model, reg_model, X, charge_threshold=0.5):
    charge_prob = clf_model.predict_proba(X)[:, 1]
    will_charge = (charge_prob >= charge_threshold).astype(int)
    magnitude = np.clip(reg_model.predict(X), 0, None)  # energy can't be negative
    return will_charge * magnitude


def evaluate(name, y_true, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)
    # mape = np.mean(np.abs((y_true + 1e-8 - pred) / (y_true + 1e-8))) * 100

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    # print(f"MAPE : {mape:.2f}%")
    print(f"R2   : {r2:.4f}")


train_pred = hurdle_predict(clf, reg, X_train)
val_pred = hurdle_predict(clf, reg, X_val)
test_pred = hurdle_predict(clf, reg, X_test)

evaluate("Train", y_train, train_pred)
evaluate("Validation", y_val, val_pred)
evaluate("Test", y_test, test_pred)



Train
----------------------------------------
MAE  : 2.8135
RMSE : 5.6035
R2   : 0.9552

Validation
----------------------------------------
MAE  : 5.2335
RMSE : 10.8699
R2   : 0.9259

Test
----------------------------------------
MAE  : 4.0973
RMSE : 9.4155
R2   : 0.9188


In [11]:
def evaluate_by_dataset(name, df, y_true, pred):
    print(f"\n{'='*60}")
    print(f"{name} — per-dataset breakdown")
    print(f"{'='*60}")

    results = []
    for dataset in sorted(df["source_dataset"].unique()):
        mask = (df["source_dataset"] == dataset).values

        y_sub = y_true[mask]
        pred_sub = pred[mask]

        mae = mean_absolute_error(y_sub, pred_sub)
        rmse = np.sqrt(mean_squared_error(y_sub, pred_sub))
        r2 = r2_score(y_sub, pred_sub)
        wape = np.abs(y_sub - pred_sub).sum() / np.abs(y_sub).sum() * 100

        results.append({
            "dataset": dataset,
            "n_rows": mask.sum(),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "WAPE%": wape,
        })

    results_df = pd.DataFrame(results).sort_values("dataset").reset_index(drop=True)
    print(results_df.to_string(index=False))
    return results_df


train_by_dataset = evaluate_by_dataset("Train", train_df, y_train.values, train_pred)
val_by_dataset = evaluate_by_dataset("Validation", val_df, y_val.values, val_pred)
test_by_dataset = evaluate_by_dataset("Test", test_df, y_test.values, test_pred)


Train — per-dataset breakdown
  dataset  n_rows      MAE      RMSE       R2     WAPE%
      acn   45860 1.532920  3.023961 0.985139 11.461660
  boulder   19791 2.151912  3.426538 0.834320 28.669299
   dundee   11575 3.082031  7.128187 0.943132 18.680158
palo_alto   57793 2.915088  4.644448 0.967817 13.123009
    paris     890 8.507317 11.405547 0.695035 25.355654
    perth   23900 5.089959  9.184492 0.842900 31.214471
      sap   21892 2.967417  7.046227 0.965203 16.575053

Validation — per-dataset breakdown
  dataset  n_rows       MAE      RMSE       R2     WAPE%
      acn    9827  0.175424  0.862314 0.959652 20.113276
  boulder    4241  1.377470  2.390969 0.744031 39.123240
   dundee    2480 18.187294 25.219342 0.714602 23.849331
palo_alto   12384  6.121998  9.096824 0.944244 13.798130
    paris     191 10.578725 13.917671 0.538798 29.175492
    perth    5122  9.651217 15.462471 0.828032 27.866776
      sap    4691  5.080334 12.261184 0.938826 21.513644

Test — per-dataset breakdown

In [12]:
def wape(y_true, pred):
    return np.abs(np.asarray(y_true) - np.asarray(pred)).sum() / np.abs(np.asarray(y_true)).sum() * 100

original_metrics = {
    "train": {
        "mae": mean_absolute_error(y_train, train_pred),
        "rmse": np.sqrt(mean_squared_error(y_train, train_pred)),
        "r2": r2_score(y_train, train_pred),
        "wape": wape(y_train, train_pred),
    },
    "val": {
        "mae": mean_absolute_error(y_val, val_pred),
        "rmse": np.sqrt(mean_squared_error(y_val, val_pred)),
        "r2": r2_score(y_val, val_pred),
        "wape": wape(y_val, val_pred),
    },
    "test": {
        "mae": mean_absolute_error(y_test, test_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, test_pred)),
        "r2": r2_score(y_test, test_pred),
        "wape": wape(y_test, test_pred),
    },
}

print("Original experiment — test set (for later comparison):")
for k, v in original_metrics["test"].items():
    print(f"  {k.upper():5s}: {v:.4f}" if k != "wape" else f"  {k.upper():5s}: {v:.2f}%")


Original experiment — test set (for later comparison):
  MAE  : 4.0973
  RMSE : 9.4155
  R2   : 0.9188
  WAPE : 21.47%


In [13]:
N_SPLITS = 5

cv_data = data.sort_values("hour").reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

cv_results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(cv_data), start=1):

    train_fold = cv_data.iloc[train_idx].copy()
    val_fold = cv_data.iloc[val_idx].copy()

    # --- entity-level stats from this fold's TRAIN portion only ---
    fold_entity_stats = (
        train_fold.groupby("entity_id")["kwh_allocated"]
        .agg(entity_mean_kwh="mean", entity_std_kwh="std", entity_max_kwh="max")
        .reset_index()
    )
    fold_entity_stats["entity_std_kwh"] = fold_entity_stats["entity_std_kwh"].fillna(0)

    train_fold = train_fold.merge(fold_entity_stats, on="entity_id", how="left")
    val_fold = val_fold.merge(fold_entity_stats, on="entity_id", how="left")

    fold_global_mean = train_fold["kwh_allocated"].mean()
    for d in [train_fold, val_fold]:
        d["entity_mean_kwh"] = d["entity_mean_kwh"].fillna(fold_global_mean)
        d["entity_std_kwh"] = d["entity_std_kwh"].fillna(0)
        d["entity_max_kwh"] = d["entity_max_kwh"].fillna(fold_global_mean)

    # --- features ---
    fold_drop_cols = ["hour", "kwh_allocated", "entity_id", "site"]

    X_train_fold = train_fold.drop(columns=fold_drop_cols)
    y_train_fold = train_fold[TARGET]

    X_val_fold = val_fold.drop(columns=fold_drop_cols)
    y_val_fold = val_fold[TARGET]

    fold_cat_cols = [c for c in ["source_dataset", "region"] if c in X_train_fold.columns]
    X_train_fold = pd.get_dummies(X_train_fold, columns=fold_cat_cols, drop_first=True)
    X_val_fold = pd.get_dummies(X_val_fold, columns=fold_cat_cols, drop_first=True)
    X_train_fold, X_val_fold = X_train_fold.align(X_val_fold, join="left", axis=1, fill_value=0)

    # --- stage A: classifier ---
    y_train_fold_binary = (y_train_fold > 0).astype(int)
    y_val_fold_binary = (y_val_fold > 0).astype(int)

    clf_fold = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )
    clf_fold.fit(
        X_train_fold, y_train_fold_binary,
        eval_set=[(X_val_fold, y_val_fold_binary)],
        verbose=False,
    )

    # --- stage B: Tweedie regressor, nonzero rows only ---
    nz_train_mask = (y_train_fold > 0).values
    nz_val_mask = (y_val_fold > 0).values

    X_train_fold_nz = X_train_fold[nz_train_mask]
    y_train_fold_nz = y_train_fold[nz_train_mask]
    X_val_fold_nz = X_val_fold[nz_val_mask]
    y_val_fold_nz = y_val_fold[nz_val_mask]

    reg_fold = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:tweedie",
        tweedie_variance_power=1.3,
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )
    reg_fold.fit(
        X_train_fold_nz, y_train_fold_nz,
        eval_set=[(X_val_fold_nz, y_val_fold_nz)],
        verbose=False,
    )

    # --- combine + score this fold ---
    val_pred_fold = hurdle_predict(clf_fold, reg_fold, X_val_fold)

    mae = mean_absolute_error(y_val_fold, val_pred_fold)
    rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred_fold))
    r2 = r2_score(y_val_fold, val_pred_fold)
    mape = np.mean(np.abs((y_val_fold + 1e-8 - val_pred_fold) / (y_val_fold + 1e-8))) * 100

    print(f"Fold {fold}: train={len(train_fold)}  val={len(val_fold)}  "
          f"MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}  MAPE={mape:.2f}%")

    cv_results.append({"fold": fold, "mae": mae, "rmse": rmse, "r2": r2, "mape": mape})

cv_results_df = pd.DataFrame(cv_results)
cv_results_df


Fold 1: train=43263  val=43263  MAE=5.7432  RMSE=10.6060  R2=0.8742  MAPE=935252243.98%
Fold 2: train=86526  val=43263  MAE=6.3418  RMSE=12.8315  R2=0.8544  MAPE=664560723.39%
Fold 3: train=129789  val=43263  MAE=5.5569  RMSE=12.2277  R2=0.8841  MAPE=636221346.81%
Fold 4: train=173052  val=43263  MAE=3.1852  RMSE=7.6631  R2=0.9570  MAPE=468772817.95%
Fold 5: train=216315  val=43263  MAE=1.9301  RMSE=5.7525  R2=0.8782  MAPE=432660846.46%


,fold,mae,rmse,r2,mape
0,1,5.743198,10.606026,0.874208,9.352522e+08
1,2,6.341846,12.831488,0.854408,6.645607e+08
2,3,5.556866,12.227670,0.884110,6.362213e+08
3,4,3.185234,7.663120,0.957045,4.687728e+08
4,5,1.930051,5.752481,0.878250,4.326608e+08


In [14]:
print("Cross-validated performance (mean +/- std across folds)")
print("-" * 55)
for metric in ["mae", "rmse", "r2", "mape"]:
    print(f"{metric.upper():5s}: {cv_results_df[metric].mean():.4f} +/- {cv_results_df[metric].std():.4f}")


Cross-validated performance (mean +/- std across folds)
-------------------------------------------------------
MAE  : 4.5514 +/- 1.8957
RMSE : 9.8162 +/- 3.0282
R2   : 0.8896 +/- 0.0393
MAPE : 627493595.7200 +/- 199573054.7020


In [15]:
data_sorted = data.sort_values("hour").reset_index(drop=True)
print("Total rows:", len(data_sorted))


Total rows: 259578


In [16]:
N_SPLITS = 5

test_start = int(len(data_sorted) * 0.85)

development_data = data_sorted.iloc[:test_start].copy()
final_test_data = data_sorted.iloc[test_start:].copy()

print("Total rows       :", len(data_sorted))
print("Development rows :", len(development_data))
print("Final test rows  :", len(final_test_data))

print("\nDevelopment period:")
print(development_data["hour"].min(), "->", development_data["hour"].max())

print("\nFinal test period:")
print(final_test_data["hour"].min(), "->", final_test_data["hour"].max())


Total rows       : 259578
Development rows : 220641
Final test rows  : 38937

Development period:
2011-08-05 20:00:00+00:00 -> 2020-06-11 06:00:00+00:00

Final test period:
2020-06-11 06:00:00+00:00 -> 2021-04-12 21:00:00+00:00


In [17]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_results = []
print(f"Running {N_SPLITS}-fold TimeSeriesSplit on development_data only ({len(development_data)} rows)")


Running 5-fold TimeSeriesSplit on development_data only (220641 rows)


In [18]:
for fold, (train_idx, val_idx) in enumerate(tscv.split(development_data), start=1):

    train_fold = development_data.iloc[train_idx].copy()
    val_fold = development_data.iloc[val_idx].copy()

    print("=" * 70)
    print(f"FOLD {fold}")
    print("Train:", len(train_fold), " Validation:", len(val_fold))

    fold_entity_stats = (
        train_fold.groupby("entity_id")["kwh_allocated"]
        .agg(entity_mean_kwh="mean", entity_std_kwh="std", entity_max_kwh="max")
        .reset_index()
    )
    fold_entity_stats["entity_std_kwh"] = fold_entity_stats["entity_std_kwh"].fillna(0)

    train_fold = train_fold.merge(fold_entity_stats, on="entity_id", how="left")
    val_fold = val_fold.merge(fold_entity_stats, on="entity_id", how="left")

    fold_global_mean = train_fold["kwh_allocated"].mean()
    for d in [train_fold, val_fold]:
        d["entity_mean_kwh"] = d["entity_mean_kwh"].fillna(fold_global_mean)
        d["entity_std_kwh"] = d["entity_std_kwh"].fillna(0)
        d["entity_max_kwh"] = d["entity_max_kwh"].fillna(fold_global_mean)

    fold_drop_cols = ["hour", "kwh_allocated", "entity_id", "site"]

    X_train_fold = train_fold.drop(columns=fold_drop_cols)
    y_train_fold = train_fold["kwh_allocated"]

    X_val_fold = val_fold.drop(columns=fold_drop_cols)
    y_val_fold = val_fold["kwh_allocated"]

    fold_cat_cols = [c for c in ["source_dataset", "region"] if c in X_train_fold.columns]
    X_train_fold = pd.get_dummies(X_train_fold, columns=fold_cat_cols, drop_first=True)
    X_val_fold = pd.get_dummies(X_val_fold, columns=fold_cat_cols, drop_first=True)
    X_train_fold, X_val_fold = X_train_fold.align(X_val_fold, join="left", axis=1, fill_value=0)

    y_train_fold_binary = (y_train_fold > 0).astype(int)
    y_val_fold_binary = (y_val_fold > 0).astype(int)

    clf_fold = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0, eval_metric="logloss",
        early_stopping_rounds=30, random_state=42, n_jobs=-1,
    )
    clf_fold.fit(
        X_train_fold, y_train_fold_binary,
        eval_set=[(X_val_fold, y_val_fold_binary)],
        verbose=False,
    )

    nz_train_mask = (y_train_fold > 0).values
    nz_val_mask = (y_val_fold > 0).values

    X_train_fold_nz = X_train_fold[nz_train_mask]
    y_train_fold_nz = y_train_fold[nz_train_mask]
    X_val_fold_nz = X_val_fold[nz_val_mask]
    y_val_fold_nz = y_val_fold[nz_val_mask]

    reg_fold = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0, objective="reg:tweedie",
        tweedie_variance_power=1.3, early_stopping_rounds=30, random_state=42, n_jobs=-1,
    )
    reg_fold.fit(
        X_train_fold_nz, y_train_fold_nz,
        eval_set=[(X_val_fold_nz, y_val_fold_nz)],
        verbose=False,
    )

    val_pred_fold = hurdle_predict(clf_fold, reg_fold, X_val_fold)

    mae = mean_absolute_error(y_val_fold, val_pred_fold)
    rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred_fold))
    r2 = r2_score(y_val_fold, val_pred_fold)
    fold_wape = wape(y_val_fold, val_pred_fold)

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"WAPE : {fold_wape:.2f}%")

    cv_results.append({"fold": fold, "mae": mae, "rmse": rmse, "r2": r2, "wape": fold_wape})

FOLD 1
Train: 36776  Validation: 36773
MAE  : 9.1647
RMSE : 16.4395
R2   : 0.6984
WAPE : 35.70%
FOLD 2
Train: 73549  Validation: 36773
MAE  : 5.5558
RMSE : 10.3230
R2   : 0.8773
WAPE : 24.79%
FOLD 3
Train: 110322  Validation: 36773
MAE  : 6.2413
RMSE : 13.1518
R2   : 0.8626
WAPE : 24.07%
FOLD 4
Train: 147095  Validation: 36773
MAE  : 4.7390
RMSE : 10.2293
R2   : 0.9263
WAPE : 19.32%
FOLD 5
Train: 183868  Validation: 36773
MAE  : 2.7581
RMSE : 7.1068
R2   : 0.9575
WAPE : 16.91%


In [19]:
cv_results_df = pd.DataFrame(cv_results)

print("CV RESULTS (development data only, 5-fold TimeSeriesSplit)")
print(cv_results_df.to_string(index=False))

print("\nMEAN +/- STD")
for metric in ["mae", "rmse", "r2", "wape"]:
    unit = "%" if metric == "wape" else ""
    print(f"{metric.upper():5s}: {cv_results_df[metric].mean():.4f}{unit} +/- {cv_results_df[metric].std():.4f}{unit}")


CV RESULTS (development data only, 5-fold TimeSeriesSplit)
 fold      mae      rmse       r2      wape
    1 9.164673 16.439550 0.698445 35.699915
    2 5.555774 10.322959 0.877334 24.793767
    3 6.241288 13.151758 0.862591 24.069435
    4 4.739022 10.229350 0.926332 19.316123
    5 2.758071  7.106848 0.957538 16.909866

MEAN +/- STD
MAE  : 5.6918 +/- 2.3396
RMSE : 11.4501 +/- 3.5148
R2   : 0.8644 +/- 0.1003
WAPE : 24.1578% +/- 7.2389%


In [20]:
final_train = development_data.copy()

entity_stats = (
    final_train
    .groupby("entity_id")["kwh_allocated"]
    .agg(
        entity_mean_kwh="mean",
        entity_std_kwh="std",
        entity_max_kwh="max"
    )
    .reset_index()
)

entity_stats["entity_std_kwh"] = entity_stats["entity_std_kwh"].fillna(0)

final_train = final_train.merge(
    entity_stats,
    on="entity_id",
    how="left"
)

global_mean = final_train["kwh_allocated"].mean()

final_train["entity_mean_kwh"] = final_train["entity_mean_kwh"].fillna(global_mean)
final_train["entity_std_kwh"] = final_train["entity_std_kwh"].fillna(0)
final_train["entity_max_kwh"] = final_train["entity_max_kwh"].fillna(global_mean)

final_test = final_test_data.merge(
    entity_stats,
    on="entity_id",
    how="left"
)

final_test["entity_mean_kwh"] = final_test["entity_mean_kwh"].fillna(global_mean)
final_test["entity_std_kwh"] = final_test["entity_std_kwh"].fillna(0)
final_test["entity_max_kwh"] = final_test["entity_max_kwh"].fillna(global_mean)

print(final_train.shape)
print(final_test.shape)

(220641, 47)
(38937, 47)


In [21]:
drop_cols = [
    "hour",
    "kwh_allocated",
    "entity_id",
    "site"
]

X_final_train = final_train.drop(
    columns=drop_cols
)

y_final_train = final_train["kwh_allocated"]

X_final_test = final_test.drop(
    columns=drop_cols
)

y_final_test = final_test["kwh_allocated"]

cat_cols = [
    c for c in ["source_dataset", "region"]
    if c in X_final_train.columns
]

X_final_train = pd.get_dummies(
    X_final_train,
    columns=cat_cols,
    drop_first=True
)

X_final_test = pd.get_dummies(
    X_final_test,
    columns=cat_cols,
    drop_first=True
)

X_final_train, X_final_test = X_final_train.align(
    X_final_test,
    join="left",
    axis=1,
    fill_value=0
)

print("Train:", X_final_train.shape)
print("Test :", X_final_test.shape)

Train: (220641, 51)
Test : (38937, 51)


In [22]:
from xgboost import XGBClassifier

y_final_binary = (y_final_train > 0).astype(int)

final_clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

final_clf.fit(
    X_final_train,
    y_final_binary,
    verbose=False
)

print("Final classifier trained.")

Final classifier trained.


In [23]:
from xgboost import XGBRegressor

positive_mask = y_final_train > 0

X_final_train_positive = X_final_train.loc[
    positive_mask
]

y_final_train_positive = y_final_train.loc[
    positive_mask
]

final_reg = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:tweedie",
    tweedie_variance_power=1.3,
    random_state=42,
    n_jobs=-1
)

final_reg.fit(
    X_final_train_positive,
    y_final_train_positive,
    verbose=False
)

print("Positive training rows:", len(y_final_train_positive))
print("Final Tweedie regressor trained.")

Positive training rows: 161006
Final Tweedie regressor trained.


In [24]:
final_pred = hurdle_predict(final_clf, final_reg, X_final_test)

print("Predictions generated.")
print("Prediction shape:", final_pred.shape)
print("Min prediction:", final_pred.min())
print("Max prediction:", final_pred.max())

Predictions generated.
Prediction shape: (38937,)
Min prediction: 0.0
Max prediction: 209.2330322265625


In [25]:
final_metrics = {
    "mae": mean_absolute_error(y_final_test, final_pred),
    "rmse": np.sqrt(mean_squared_error(y_final_test, final_pred)),
    "r2": r2_score(y_final_test, final_pred),
    "wape": wape(y_final_test, final_pred),
}

print("=" * 60)
print("FINAL UNTOUCHED TEST RESULTS (corrected protocol, original prediction rule)")
print("=" * 60)
print(f"MAE  : {final_metrics['mae']:.4f}")
print(f"RMSE : {final_metrics['rmse']:.4f}")
print(f"R2   : {final_metrics['r2']:.4f}")
print(f"WAPE : {final_metrics['wape']:.2f}%")


FINAL UNTOUCHED TEST RESULTS (corrected protocol, original prediction rule)
MAE  : 1.9294
RMSE : 5.6746
R2   : 0.8889
WAPE : 26.42%


In [26]:
comparison_df = pd.DataFrame([
    {
        "Experiment": "Original (70/15/15 per-entity split)",
        "Test R2": original_metrics["test"]["r2"],
        "MAE": original_metrics["test"]["mae"],
        "RMSE": original_metrics["test"]["rmse"],
        "WAPE%": original_metrics["test"]["wape"],
        "Prediction Method": "hard threshold (>=0.5) x magnitude",
    },
    {
        "Experiment": "Corrected (85% dev CV / 15% untouched final test)",
        "Test R2": final_metrics["r2"],
        "MAE": final_metrics["mae"],
        "RMSE": final_metrics["rmse"],
        "WAPE%": final_metrics["wape"],
        "Prediction Method": "hard threshold (>=0.5) x magnitude",
    },
])

comparison_df


,Experiment,Test R2,MAE,RMSE,WAPE%,Prediction Method
0,Original (70/15/15 per-entity split),0.918758,4.097346,9.415501,21.466834,hard threshold (>=0.5) x magnitude
1,Corrected (85% dev CV / 15% untouched final test),0.888861,1.929442,5.674616,26.416458,hard threshold (>=0.5) x magnitude


In [27]:
df = df.sort_values(["source_dataset", "entity_id", "hour"])

df["target_t_plus_1"] = (
    df.groupby(["source_dataset", "entity_id"])["kwh_allocated"]
      .shift(-1)
)

df = df.dropna(subset=["target_t_plus_1"])

In [28]:
y = df["target_t_plus_1"]